In [3]:
from io import BytesIO
import urllib.request
import zipfile
import pandas as pd
from sqlalchemy import create_engine

# Web上のZIPファイルのURL
zip_url = "https://www.mhlw.go.jp/content/11121000/02-2_clinic_speciality_hours_20250601.zip"

print("WebからZIPファイルをダウンロード中")

# Web上のZIPファイルをメモリ（パソコンの頭の中）に一時保存する
# BytesIOを使うことで、パソコンのハードディスクに保存せず、直接中身を覗ける
with urllib.request.urlopen(zip_url) as response:
    zip_data = BytesIO(response.read())

print("ZIPファイルを解凍して、CSVを読み込み")
# ZIPファイルを開いて、中にあるCSVファイルをPandasで読み込む
with zipfile.ZipFile(zip_data) as z:
    # ZIPの中に入っている「ファイル名のリスト」を取得する
    file_list = z.namelist()
    # 1番目のファイル（目的のCSV）の名前を自動で取得する
    # もしZIPの中に複数のCSVがあるなら、file_list[0] の数字を変えるか直接名前を書く
    target_csv_name = file_list[0] 
    
    # ZIP内のCSVファイルを直接Pandasに渡して読み込む！
    with z.open(target_csv_name) as f:
        df = pd.read_csv(f)

print(f"読み込み成功！ データの件数: {len(df)} 件 / カラム数: {len(df.columns)}")

# --- MySQLへ直接流し込む ---

# MySQLへの「直通トンネル（エンジン）」を作る
engine = create_engine('mysql+pymysql://root:@localhost/example')

# 5. 【一撃！】MySQLへデータを直接流し込む
table_name = "gp_time" # 作りたいテーブルの名前
df.to_sql(name=table_name, con=engine, if_exists='append', index=False)

print(f"完了！ Web上のZIPから直接 MySQLの '{table_name}' へデータを保存")

WebからZIPファイルをダウンロード中
ZIPファイルを解凍して、CSVを読み込み


C:\Users\user\AppData\Local\Temp\ipykernel_5848\4126022689.py:28: DtypeWarning: Columns (0: ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


読み込み成功！ データの件数: 613221 件 / カラム数: 36
完了！ Web上のZIPから直接 MySQLの 'gp_time' へデータを保存


In [10]:
engine = create_engine('mysql+pymysql://root:@localhost/example')

# DESCRIBEコマンドでテーブルの構造を確認する
sql_query = "DESCRIBE gp_time;"

# 実行してPandasで見てみよう
df_structure = pd.read_sql(sql_query, con=engine)
df_structure

,Field,Type,Null,Key,Default,Extra
0,ID,text,YES,,None,
1,診療科目コード,bigint(20),YES,,None,
2,診療科目名,text,YES,,None,
3,診療時間帯,bigint(20),YES,,None,
4,月_診療開始時間,text,YES,,None,
5,月_診療終了時間,text,YES,,None,
6,火_診療開始時間,text,YES,,None,
7,火_診療終了時間,text,YES,,None,
8,水_診療開始時間,text,YES,,None,
9,水_診療終了時間,text,YES,,None,


In [11]:
#  欲しいカラム名が詰まったリスト
select_columns = ['診療科目名', '月_診療開始時間', '月_診療終了時間']

# カンマとバッククォートで綺麗に合体させる
# ※実務ではカラム名の前後に ``（バッククォート）をつけるとSQLが安全に動くよ
columns_sql = ", ".join([f"`{col}`" for col in select_columns])

# 3. 変換された文字列を確認してみよう
print(columns_sql)
# ➔ 画面には `診療科目名`, `月_診療開始時間`, `月_診療終了時間` と表示される！

# 4. あとは f文字列（f-string）を使って SQL に埋め込むだけ！
sql_query = f"SELECT {columns_sql} FROM mymemo_table;"
print(sql_query)
# ➔ SELECT `診療科目名`, `月_診療開始時間`, `月_診療終了時間` FROM mymemo_table; という完璧なSQLの完成！

`診療科目名`, `月_診療開始時間`, `月_診療終了時間`
SELECT `診療科目名`, `月_診療開始時間`, `月_診療終了時間` FROM mymemo_table;


In [ ]:
# さっきの DESCRIBE で取ってきた設計図の df があるとする
# columns_list = df_structure['Field'].tolist() で全カラム名を取得したとします
all_columns = ['ID', '診療科目名', '月_診療開始時間', '月_診療終了時間', '火_診療開始時間', '火_診療終了時間']

# ➔ 「月_」という文字が含まれるカラムだけを、Pythonに自動で選ばせる！
monday_columns = [col for col in all_columns if "月_" in col]
# monday_columns の中身は自動的に ['月_診療開始時間', '月_診療終了時間'] になるよ！

# ➔ あとはさっきの .join で合体！
columns_sql = ", ".join([f"`{col}` for col in monday_columns])
sql_query = f"SELECT {columns_sql} FROM gp_time;"
print(sql_query)
# ➔ SELECT `月_診療開始時間`, `月_診療終了時間` FROM mymemo_table; が全自動で作られた！

SELECT `月_診療開始時間`, `月_診療終了時間` FROM gp_time;


In [8]:
# 1. 倉庫（MySQL）への直通トンネルを開く
engine = create_engine('mysql+pymysql://root:@localhost/example')

# 2. 【ここがプロの技】SQLを使って、欲しいデータ「だけ」をPandasに呼び出す！
# 例：20代のデータだけ、特定のカラム（petal）だけ、などを指定してスマートに引き出す
sql_query = "SELECT ID FROM gp_time;"

# 3. read_sql を使えば、SQLの結果がそのまま綺麗な df（Pandas）になって出てくる！
df_selected = pd.read_sql(sql_query, con=engine)
df_selected.head()  # 最初の5行だけ見てみよう


,ID
0,120116711805
1,120116711805
2,120116711805
3,120116711805
4,120116711805


In [ ]:
# 「データを0件だけ取ってくる」という条件にする
sql_query = "SELECT * FROM gp_time LIMIT 0;"

df_empty = pd.read_sql(sql_query, con=engine)

# .columns を見れば、カラム名だけのリストが手に入る！
print(df_empty.columns.tolist())

['ID', '診療科目コード', '診療科目名', '診療時間帯', '月_診療開始時間', '月_診療終了時間', '火_診療開始時間', '火_診療終了時間', '水_診療開始時間', '水_診療終了時間', '木_診療開始時間', '木_診療終了時間', '金_診療開始時間', '金_診療終了時間', '土_診療開始時間', '土_診療終了時間', '日_診療開始時間', '日_診療終了時間', '祝_診療開始時間', '祝_診療終了時間', '月_外来受付開始時間', '月_外来受付終了時間', '火_外来受付開始時間', '火_外来受付終了時間', '水_外来受付開始時間', '水_外来受付終了時間', '木_外来受付開始時間', '木_外来受付終了時間', '金_外来受付開始時間', '金_外来受付終了時間', '土_外来受付開始時間', '土_外来受付終了時間', '日_外来受付開始時間', '日_外来受付終了時間', '祝_外来受付開始時間', '祝_外来受付終了時間']
